In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from ase import Atoms
from ase.io import read, write

In [3]:
input_file = "Fe_E111_110_slabbed_min.lmp"

input_dir = Path(f"../input").resolve()
input_path = input_dir / input_file

atoms = read(input_path, format="lammps-data")

In [8]:
num_y_layers = 70
num_fixed_layers = 1
prec_radius = 20

output_file = f"Fe_E111_110_R{prec_radius}.lmp"

In [9]:
# -------------------------
# 1. Periodicity and Vacuum
# -------------------------
atoms.set_pbc([True, False, True])
atoms.center(vacuum=5.0, axis=1)

# -------------------------
# 2. Shift atoms by half box length in X
# -------------------------
Lx = atoms.cell[0, 0]
atoms.positions[:, 0] -= Lx / 2.0

# -------------------------
# 3. Y-layer logic for top/bottom slabs
# -------------------------
y = atoms.positions[:, 1]
y_min, y_max = y.min(), y.max()
layer_thickness = (y_max - y_min) / num_y_layers
layer_index = ((y - y_min) / layer_thickness).astype(int)
layer_index = np.clip(layer_index, 0, num_y_layers - 1)

# -------------------------
# 4. Initialize types
# -------------------------
types = np.ones(len(atoms), dtype=int)  # default = 1 (Middle/Mobile region)

# Bottom slabs -> type 3 (lower layer indices)
bottom_mask = (layer_index < num_fixed_layers)
types[bottom_mask] = 3

# Top slabs -> type 2 (higher layer indices)
top_mask = (layer_index >= num_y_layers - num_fixed_layers)
types[top_mask] = 2

# -------------------------
# 5. Precipitate spherical region -> type 4 (handles PBC)
# -------------------------
# Fractional coordinates
frac = atoms.get_scaled_positions()  # [0,1)
center_frac = np.array([0.5, 0.5, 0.5])  # box center

# Minimum image convention for periodic boundaries
delta = frac - center_frac
delta -= np.round(delta)  # wrap into [-0.5, 0.5]

# Convert to Cartesian distances
cart_delta = delta * atoms.get_cell().diagonal()
distances = np.linalg.norm(cart_delta, axis=1)

# Assign precipitate type 4
# Note: This will overwrite slab types if the sphere overlaps them.
sphere_mask = distances <= prec_radius
types[sphere_mask] = 4

# -------------------------
# 6. Attach types and wrap positions
# -------------------------
atoms.set_array("type", types)
atoms.wrap()

# -------------------------
# 7. Write LAMMPS data file
# -------------------------
unique_types = np.unique(types)
write(
    input_dir / output_file,
    atoms,
    format="lammps-data",
    atom_style="atomic",
    specorder=[str(t) for t in unique_types]  # ensures header matches actual types
)